In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
from PIL import Image

# Save directly to Drive during download
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"
os.makedirs(DRIVE_IMAGE_DIR, exist_ok=True)

# Check how many already downloaded in case of partial download
already_downloaded = set(
    f.replace("_rgb.png", "")
    for f in os.listdir(DRIVE_IMAGE_DIR)
    if f.endswith(".png")
)
print(f"Already in Drive: {len(already_downloaded)}")

MAX_IMAGES = 5000
count = len(already_downloaded)

bucket_base_path = "gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead/"

dish_folders = subprocess.check_output(
    f"gsutil ls {bucket_base_path}", shell=True
).decode().splitlines()

print(f"Found {len(dish_folders)} dish folders.")
print(f"Starting from image {count + 1}...")

for folder in dish_folders:
    if count >= MAX_IMAGES:
        break

    folder_name = folder.rstrip('/').split('/')[-1]

    if folder_name in already_downloaded:
        continue

    remote_rgb_path = folder + "rgb.png"
    local_tmp_path = f"/tmp/{folder_name}_rgb.png"
    drive_path = os.path.join(DRIVE_IMAGE_DIR, f"{folder_name}_rgb.png")

    result = subprocess.run(
        f"gsutil cp {remote_rgb_path} {local_tmp_path}",
        shell=True, capture_output=True
    )

    if result.returncode == 0:
        try:
            img = Image.open(local_tmp_path).convert("RGB")
            img = img.resize((224, 224))
            img.save(drive_path)
            os.remove(local_tmp_path)
            count += 1
            if count % 100 == 0:
                print(f"{count} images saved to Drive...")
        except Exception as e:
            print(f"Error processing {folder_name}: {e}")
    else:
        print(f"Failed: {remote_rgb_path}")

print(f"Download complete: {count} total images in Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already in Drive: 3089
Found 3490 dish folders.
Starting from image 3090...
3100 images saved to Drive...
3200 images saved to Drive...
3300 images saved to Drive...
3400 images saved to Drive...
Download complete: 3490 total images in Drive.


In [ ]:
import os
import csv
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

DRIVE_IMAGE_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"

downloaded_dish_ids = set(
    f.replace("_rgb.png", "")
    for f in os.listdir(DRIVE_IMAGE_DIR)
    if f.endswith(".png")
)
print(f"Total images in Drive: {len(downloaded_dish_ids)}")

dish_level_rows = []
for csv_path in ["/content/dish_metadata_cafe1.csv",
                 "/content/dish_metadata_cafe2.csv"]:
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) >= 6:
                dish_level_rows.append(row[:6])

dish_level_all = pd.DataFrame(
    dish_level_rows,
    columns=["dish_id", "total_calories", "total_mass",
             "total_fat", "total_carb", "total_protein"]
)

for col in ["total_calories", "total_mass", "total_fat",
            "total_carb", "total_protein"]:
    dish_level_all[col] = pd.to_numeric(
        dish_level_all[col], errors='coerce'
    )

dish_level_all['dish_id'] = dish_level_all['dish_id'].astype(str).str.strip()

filtered = dish_level_all[dish_level_all['dish_id'].isin(downloaded_dish_ids)]
filtered = filtered.dropna()

print(f"Rows after filtering: {len(filtered)}")

filtered.to_csv("/content/drive/MyDrive/labels_all.csv", index=False)
filtered.to_csv("/content/labels_all.csv", index=False)
print("labels_all.csv saved!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total images in Drive: 3490
Rows after filtering: 3490
labels_all.csv saved!


In [ ]:
import os
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

DRIVE_IMAGE_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"

all_files = [f for f in os.listdir(DRIVE_IMAGE_DIR) if f.endswith(".png")]
print(f"Total files in folder: {len(all_files)}")

good = 0
bad = 0
for file in all_files:
    try:
        img = Image.open(os.path.join(DRIVE_IMAGE_DIR, file))
        img.verify()
        good += 1
    except Exception:
        bad += 1

print(f"Good images: {good}")
print(f"Bad images: {bad}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total files in folder: 3490
Good images: 3489
Bad images: 1


In [ ]:
import os
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

DRIVE_IMAGE_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"
CSV_FILE = "/content/drive/MyDrive/labels_all.csv"

df = pd.read_csv(CSV_FILE)
print(f"Rows before cleaning: {len(df)}")

# Find and remove the 1 bad image
bad_images = []
for file in os.listdir(DRIVE_IMAGE_DIR):
    if file.endswith(".png"):
        try:
            img = Image.open(os.path.join(DRIVE_IMAGE_DIR, file))
            img.verify()
        except Exception:
            bad_images.append(file)
            os.remove(os.path.join(DRIVE_IMAGE_DIR, file))
            print(f"Removed bad image: {file}")

# Remove that dish from CSV
bad_ids = set(f.replace("_rgb.png", "") for f in bad_images)
df_clean = df[~df['dish_id'].isin(bad_ids)]
print(f"Rows after cleaning: {len(df_clean)}")

# Save cleaned CSV to both places
df_clean.to_csv("/content/drive/MyDrive/labels_all.csv", index=False)
df_clean.to_csv("/content/labels_all.csv", index=False)
print("CSV updated!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows before cleaning: 3490
Removed bad image: dish_1567106825_rgb.png
Rows after cleaning: 3489
CSV updated!


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from google.colab import drive

drive.mount('/content/drive')

CSV_FILE = "/content/labels_all.csv"
IMG_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"
SAVE_DIR = "/content/drive/MyDrive/nutrition_checkpoints_all"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}", flush=True)

class NutritionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.data[['total_calories', 'total_fat',
                                  'total_carb', 'total_protein']].values
        self.filenames = self.data['dish_id'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.filenames[idx] + "_rgb.png"
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label, self.filenames[idx]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_df = pd.read_csv(CSV_FILE)

# 5 images held out for demo prediction — model never sees these
demo_df   = full_df.iloc[:5]
remaining = full_df.iloc[5:].reset_index(drop=True)

train_size = int(0.8 * len(remaining))
train_df   = remaining.iloc[:train_size]
test_df    = remaining.iloc[train_size:]

print(f"Train: {len(train_df)}, Test: {len(test_df)}, Demo: {len(demo_df)}", flush=True)

train_dataset = NutritionDataset(train_df, IMG_DIR, transform=train_transform)
test_dataset  = NutritionDataset(test_df,  IMG_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)
model = model.to(device)
print("Model loaded successfully!", flush=True)

# Check for existing checkpoint and resume
checkpoint_path = None
for i in range(20, 0, -1):
    path = f"{SAVE_DIR}/nutrition_model_epoch{i}.pth"
    if os.path.exists(path):
        checkpoint_path = path
        start_epoch = i
        break

if checkpoint_path:
    model.load_state_dict(torch.load(checkpoint_path))
    print(f"Resumed from epoch {start_epoch}", flush=True)
else:
    start_epoch = 0
    print("Starting fresh training", flush=True)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

def train_model(model, dataloader, criterion, optimizer,
                total_epochs=20, start_epoch=0):
    model.train()
    for epoch in range(start_epoch, total_epochs):
        running_loss = 0.0
        for i, (inputs, labels, _) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            if i % 10 == 0:
                print(f"  Epoch {epoch+1}, Batch {i}/{len(dataloader)}", flush=True)

        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{total_epochs}, Loss: {epoch_loss:.4f}", flush=True)

        save_path = f"{SAVE_DIR}/nutrition_model_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved: {save_path}", flush=True)

def evaluate_model(model, dataloader):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels, _ in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    mse = np.mean((all_preds - all_labels) ** 2)
    mae = np.mean(np.abs(all_preds - all_labels), axis=0)

    print(f"\nTest MSE: {mse:.4f}", flush=True)
    print(f"MAE per nutrient:", flush=True)
    print(f"  Calories: {mae[0]:.2f}", flush=True)
    print(f"  Fat:      {mae[1]:.2f}g", flush=True)
    print(f"  Carbs:    {mae[2]:.2f}g", flush=True)
    print(f"  Protein:  {mae[3]:.2f}g", flush=True)

def predict_demo_images(model):
    model.eval()
    print("\nPredictions on held-out demo images (never seen by model):")
    print(f"{'Dish':<25} {'Cal(P)':<10} {'Cal(A)':<10} {'Fat(P)':<8} {'Fat(A)':<8} {'Carbs(P)':<10} {'Carbs(A)':<10} {'Pro(P)':<8} {'Pro(A)':<8}")
    print("-" * 100)
    for _, row in demo_df.iterrows():
        img_path = os.path.join(IMG_DIR, row['dish_id'] + "_rgb.png")
        image = Image.open(img_path).convert("RGB")
        image = test_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(image).cpu().numpy()[0]
        print(f"{row['dish_id']:<25} "
              f"{pred[0]:<10.1f} {row['total_calories']:<10.1f} "
              f"{pred[1]:<8.1f} {row['total_fat']:<8.1f} "
              f"{pred[2]:<10.1f} {row['total_carb']:<10.1f} "
              f"{pred[3]:<8.1f} {row['total_protein']:<8.1f}")

def predict_one(model, dish_id):
    model.eval()
    img_path = os.path.join(IMG_DIR, dish_id + "_rgb.png")
    image = Image.open(img_path).convert("RGB")
    image = test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(image).cpu().numpy()[0]
    actual = full_df[full_df['dish_id'] == dish_id].iloc[0]
    print(f"\nResults for {dish_id}:")
    print(f"{'Nutrient':<12} {'Predicted':>12} {'Actual':>12} {'Difference':>12}")
    print("-" * 50)
    print(f"{'Calories':<12} {pred[0]:>12.2f} {actual['total_calories']:>12.2f} {abs(pred[0]-actual['total_calories']):>12.2f}")
    print(f"{'Fat':<12} {pred[1]:>12.2f} {actual['total_fat']:>12.2f} {abs(pred[1]-actual['total_fat']):>12.2f}")
    print(f"{'Carbs':<12} {pred[2]:>12.2f} {actual['total_carb']:>12.2f} {abs(pred[2]-actual['total_carb']):>12.2f}")
    print(f"{'Protein':<12} {pred[3]:>12.2f} {actual['total_protein']:>12.2f} {abs(pred[3]-actual['total_protein']):>12.2f}")

# Run everything
train_model(model, train_loader, criterion, optimizer,
            total_epochs=20, start_epoch=start_epoch)
evaluate_model(model, test_loader)
predict_demo_images(model)
predict_one(model, "dish_1561662216")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Train: 2787, Test: 697, Demo: 5
Model loaded successfully!
Starting fresh training
  Epoch 1, Batch 0/88
  Epoch 1, Batch 10/88
  Epoch 1, Batch 20/88
  Epoch 1, Batch 30/88
  Epoch 1, Batch 40/88
  Epoch 1, Batch 50/88
  Epoch 1, Batch 60/88
  Epoch 1, Batch 70/88
  Epoch 1, Batch 80/88
Epoch 1/20, Loss: 27154.3051
Saved: /content/drive/MyDrive/nutrition_checkpoints_all/nutrition_model_epoch1.pth
  Epoch 2, Batch 0/88
  Epoch 2, Batch 10/88
  Epoch 2, Batch 20/88
  Epoch 2, Batch 30/88
  Epoch 2, Batch 40/88
  Epoch 2, Batch 50/88
  Epoch 2, Batch 60/88
  Epoch 2, Batch 70/88
  Epoch 2, Batch 80/88
Epoch 2/20, Loss: 18846.9004
Saved: /content/drive/MyDrive/nutrition_checkpoints_all/nutrition_model_epoch2.pth
  Epoch 3, Batch 0/88
  Epoch 3, Batch 10/88
  Epoch 3, Batch 20/88
  Epoch 3, Batch 30/88
  Epoch 3, Batch 40/88
  Epoch 3, Batch 50

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

os.rename(
    "/content/drive/MyDrive/nutrition_checkpoints_all",
    "/content/drive/MyDrive/nutrition_checkpoints_all_attempt1"
)
print("Done!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Done!


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from google.colab import drive

drive.mount('/content/drive')

CSV_FILE = "/content/labels_all.csv"
IMG_DIR = "/content/drive/MyDrive/nutrition5k_all_resized"
SAVE_DIR = "/content/drive/MyDrive/nutrition_checkpoints_all"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}", flush=True)

class NutritionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.data[['total_calories', 'total_fat',
                                  'total_carb', 'total_protein']].values
        self.filenames = self.data['dish_id'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.filenames[idx] + "_rgb.png"
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label, self.filenames[idx]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_df = pd.read_csv(CSV_FILE)

# 5 images held out for demo prediction — model never sees these
demo_df   = full_df.iloc[:5]
remaining = full_df.iloc[5:].reset_index(drop=True)

train_size = int(0.8 * len(remaining))
train_df   = remaining.iloc[:train_size]
test_df    = remaining.iloc[train_size:]

print(f"Train: {len(train_df)}, Test: {len(test_df)}, Demo: {len(demo_df)}", flush=True)

train_dataset = NutritionDataset(train_df, IMG_DIR, transform=train_transform)
test_dataset  = NutritionDataset(test_df,  IMG_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)
model = model.to(device)
print("Model loaded successfully!", flush=True)

# Check for existing checkpoint and resume
checkpoint_path = None
for i in range(20, 0, -1):
    path = f"{SAVE_DIR}/nutrition_model_epoch{i}.pth"
    if os.path.exists(path):
        checkpoint_path = path
        start_epoch = i
        break

if checkpoint_path:
    model.load_state_dict(torch.load(checkpoint_path))
    print(f"Resumed from epoch {start_epoch}", flush=True)
else:
    start_epoch = 0
    print("Starting fresh training", flush=True)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_model(model, dataloader, criterion, optimizer,
                total_epochs=20, start_epoch=0):
    model.train()
    for epoch in range(start_epoch, total_epochs):
        running_loss = 0.0
        for i, (inputs, labels, _) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            if i % 10 == 0:
                print(f"  Epoch {epoch+1}, Batch {i}/{len(dataloader)}", flush=True)

        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{total_epochs}, Loss: {epoch_loss:.4f}", flush=True)

        save_path = f"{SAVE_DIR}/nutrition_model_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved: {save_path}", flush=True)

def evaluate_model(model, dataloader):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels, _ in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    mse = np.mean((all_preds - all_labels) ** 2)
    mae = np.mean(np.abs(all_preds - all_labels), axis=0)

    print(f"\nTest MSE: {mse:.4f}", flush=True)
    print(f"MAE per nutrient:", flush=True)
    print(f"  Calories: {mae[0]:.2f}", flush=True)
    print(f"  Fat:      {mae[1]:.2f}g", flush=True)
    print(f"  Carbs:    {mae[2]:.2f}g", flush=True)
    print(f"  Protein:  {mae[3]:.2f}g", flush=True)

def predict_demo_images(model):
    model.eval()
    print("\nPredictions on held-out demo images (never seen by model):")
    print(f"{'Dish':<25} {'Cal(P)':<10} {'Cal(A)':<10} {'Fat(P)':<8} {'Fat(A)':<8} {'Carbs(P)':<10} {'Carbs(A)':<10} {'Pro(P)':<8} {'Pro(A)':<8}")
    print("-" * 100)
    for _, row in demo_df.iterrows():
        img_path = os.path.join(IMG_DIR, row['dish_id'] + "_rgb.png")
        image = Image.open(img_path).convert("RGB")
        image = test_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(image).cpu().numpy()[0]
        print(f"{row['dish_id']:<25} "
              f"{pred[0]:<10.1f} {row['total_calories']:<10.1f} "
              f"{pred[1]:<8.1f} {row['total_fat']:<8.1f} "
              f"{pred[2]:<10.1f} {row['total_carb']:<10.1f} "
              f"{pred[3]:<8.1f} {row['total_protein']:<8.1f}")

def predict_one(model, dish_id):
    model.eval()
    img_path = os.path.join(IMG_DIR, dish_id + "_rgb.png")
    image = Image.open(img_path).convert("RGB")
    image = test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(image).cpu().numpy()[0]
    actual = full_df[full_df['dish_id'] == dish_id].iloc[0]
    print(f"\nResults for {dish_id}:")
    print(f"{'Nutrient':<12} {'Predicted':>12} {'Actual':>12} {'Difference':>12}")
    print("-" * 50)
    print(f"{'Calories':<12} {pred[0]:>12.2f} {actual['total_calories']:>12.2f} {abs(pred[0]-actual['total_calories']):>12.2f}")
    print(f"{'Fat':<12} {pred[1]:>12.2f} {actual['total_fat']:>12.2f} {abs(pred[1]-actual['total_fat']):>12.2f}")
    print(f"{'Carbs':<12} {pred[2]:>12.2f} {actual['total_carb']:>12.2f} {abs(pred[2]-actual['total_carb']):>12.2f}")
    print(f"{'Protein':<12} {pred[3]:>12.2f} {actual['total_protein']:>12.2f} {abs(pred[3]-actual['total_protein']):>12.2f}")

# Run everything
train_model(model, train_loader, criterion, optimizer,
            total_epochs=25, start_epoch=start_epoch)
evaluate_model(model, test_loader)
predict_demo_images(model)
predict_one(model, "dish_1561662216")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Train: 2787, Test: 697, Demo: 5
Model loaded successfully!
Starting fresh training
  Epoch 1, Batch 0/88
  Epoch 1, Batch 10/88
  Epoch 1, Batch 20/88
  Epoch 1, Batch 30/88
  Epoch 1, Batch 40/88
  Epoch 1, Batch 50/88
  Epoch 1, Batch 60/88
  Epoch 1, Batch 70/88
  Epoch 1, Batch 80/88
Epoch 1/25, Loss: 14175.3214
Saved: /content/drive/MyDrive/nutrition_checkpoints_all/nutrition_model_epoch1.pth
  Epoch 2, Batch 0/88
  Epoch 2, Batch 10/88
  Epoch 2, Batch 20/88
  Epoch 2, Batch 30/88
  Epoch 2, Batch 40/88
  Epoch 2, Batch 50/88
  Epoch 2, Batch 60/88
  Epoch 2, Batch 70/88
  Epoch 2, Batch 80/88
Epoch 2/25, Loss: 5490.2261
Saved: /content/drive/MyDrive/nutrition_checkpoints_all/nutrition_model_epoch2.pth
  Epoch 3, Batch 0/88
  Epoch 3, Batch 10/88
  Epoch 3, Batch 20/88
  Epoch 3, Batch 30/88
  Epoch 3, Batch 40/88
  Epoch 3, Batch 50/